# Cost and Emissions Model (Note S4)

Transcription of the Cl2-GPAO cost and emissions model. All equations from the supplementary note are implemented below.  
**Assumptions:** Lighting + Cl2 generation only (lower bound). Capital, building, air circulation, acid gas scrubbers neglected.

In [3]:
# =============================================================================
# INPUTS
# =============================================================================

# ----- Lighting / photons -----
eta_Cl2_abs = 0.90
kJ_per_mol_photons_365nm = 340
mol_Cl_dot_per_photons = 2
mol_CH4_per_tCH4 = 62340
eta_LED = 0.60
kJ_per_kWh = 3600

# ----- Costs -----
c_LED_levelized = 0.074
c_electricity = 0.080

# ----- Chlorine (chlor-alkali, low-end) -----
kWh_per_tCl2 = 2100
mol_Cl2_per_tCl2 = 14100
c_Cl2_non_electrical = 69

# ----- Emissions -----
kgCO2e_per_kWh_grid = 0.373
kgCO2e_per_kWh_solar = 0.045
kgCO2e_per_kWh_LED_embodied = 0.265
tCO2e_per_tCl2 = 0.71 #1.11

# ----- Eqn S29 (specific energy) -----
M_CH4 = 16.04e-3
mol_per_m3_at_STP = 40.87

# ----- Scenario: mass of CH4 oxidized [tCH4] -----
T_CH4 = 1.0

In [5]:
# =============================================================================
# EQUATIONS
# =============================================================================

def runCostModel(eta_Cl = 0.1, eta_Cl2 = 0.1):

    # S12: E_lighting = T_CH4 / eta_Cl / (eta_LED * kJ_per_kWh * eta_Cl2_abs * (1/kJ_per_mol_photons_365nm) * mol_Cl_dot_per_photons * (1/mol_CH4_per_tCH4))
    E_lighting = T_CH4 / eta_Cl / (eta_LED * kJ_per_kWh * eta_Cl2_abs * (1 / kJ_per_mol_photons_365nm) * mol_Cl_dot_per_photons * (1 / mol_CH4_per_tCH4))
    # print("E_lighting: ", E_lighting)

    # S13–S14: C_lighting = E_lighting * (c_LED_levelized + c_electricity)
    C_lighting = E_lighting * (c_LED_levelized + c_electricity)
    # print("C_lighting: ", C_lighting)

    # S15: T_Cl2 = T_CH4 / eta_Cl2 * (mol_CH4_per_tCH4 / mol_Cl2_per_tCl2)
    T_Cl2 = T_CH4 / eta_Cl2 * (mol_CH4_per_tCH4 / mol_Cl2_per_tCl2)

    # S16: E_Cl2 = T_Cl2 * kWh_per_tCl2
    E_Cl2 = T_Cl2 * kWh_per_tCl2
    # print("E_Cl2: ", E_Cl2)

    # S18: C_Cl2 = E_Cl2 * c_electricity + T_Cl2 * c_Cl2_non_electrical
    C_Cl2 = E_Cl2 * c_electricity + T_Cl2 * c_Cl2_non_electrical
    # print("C_Cl2: ", C_Cl2)

    # S20–S21: C_total = C_lighting + C_Cl2
    C_total = C_lighting + C_Cl2
    print("Ctotal: ", round(C_total))

    # S22: GHG_lighting = E_lighting * (kgCO2e_per_kWh_grid + kgCO2e_per_kWh_LED_embodied) / 1000
    GHG_lighting = E_lighting * (kgCO2e_per_kWh_grid + kgCO2e_per_kWh_LED_embodied) / 1000
    # print("GHG_lighting: ", GHG_lighting)

    # S24: GHG_Cl2 = T_Cl2 * (kWh_per_tCl2 * tCO2e_per_kWh + tCO2e_per_tCl2)
    tCO2e_per_kWh = kgCO2e_per_kWh_grid / 1000
    GHG_Cl2 = T_Cl2 * (kWh_per_tCl2 * tCO2e_per_kWh + tCO2e_per_tCl2)
    # print("GHG_Cl2: ", GHG_Cl2)

    # S26: GHG_total = GHG_lighting + GHG_Cl2
    GHG_total = GHG_lighting + GHG_Cl2
    # print("Gtotal: ", round(GHG_total))

    # S27–S28: E_total = E_lighting + E_Cl2
    E_total = E_lighting + E_Cl2
    print("Etotal: ", round(E_total))

    GHGgrid = kgCO2e_per_kWh_grid/1000 * E_total
    print("GHGgrid: ", round(GHGgrid))

    GHGsolar = kgCO2e_per_kWh_solar/1000 * E_total
    print("GHGsolar: ", round(GHGsolar,1))

    # S29: specific energy [kWh/gCH4] — set P_load, delta_CH4_ppm, Q_m3_s then uncomment:
    # e = P_load / (delta_CH4_ppm * 1e-6 * mol_per_m3_at_STP * Q_m3_s * M_CH4 * 1000 * 3600)

# BASE CASE
print("--------BASE CASE actual--------")
runCostModel(eta_Cl = 0.3400770315495919, eta_Cl2 = 0.5802004772186176)

# Atmosphere
print("--------Atmosphere--------")
runCostModel(eta_Cl = 0.19809433840976903, eta_Cl2 = 0.3385297113752122)

# Dairy barn
print("--------Dairy barn--------")
runCostModel(eta_Cl = 0.29140520520881735, eta_Cl2 = 0.5013902509758703)

# WWTP
print("--------WWTP--------")
runCostModel(eta_Cl = 0.18739147717331442, eta_Cl2 = 0.28957043185048464)

# Cl2 SWEEP
print("-----------Cl2 SWEEP-----------")
eta_Cl_list = [0.36, 0.36, 0.36, 0.3, 0.27, 0.19]  
eta_Cl2_list = [0.57, 0.62, 0.62, 0.51, 0.47, 0.33]
for i in range(len(eta_Cl_list)):
    runCostModel(eta_Cl = eta_Cl_list[i], eta_Cl2 = eta_Cl2_list[i])

--------BASE CASE actual--------
Ctotal:  4275
Etotal:  32033
GHGgrid:  12
GHGsolar:  1.4
--------Atmosphere--------
Ctotal:  7333
Etotal:  54946
GHGgrid:  20
GHGsolar:  2.5
--------Dairy barn--------
Ctotal:  4971
Etotal:  37226
GHGgrid:  14
GHGsolar:  1.7
--------WWTP--------
Ctotal:  8099
Etotal:  61155
GHGgrid:  23
GHGsolar:  2.8
-----------Cl2 SWEEP-----------
Ctotal:  4170
Etotal:  31432
GHGgrid:  12
GHGsolar:  1.4
Ctotal:  4022
Etotal:  30118
GHGgrid:  11
GHGsolar:  1.4
Ctotal:  4022
Etotal:  30118
GHGgrid:  11
GHGsolar:  1.4
Ctotal:  4853
Etotal:  36377
GHGgrid:  14
GHGsolar:  1.6
Ctotal:  5339
Etotal:  39946
GHGgrid:  15
GHGsolar:  1.8
Ctotal:  7594
Etotal:  56828
GHGgrid:  21
GHGsolar:  2.6
